# 04 - Collapse prediction model (SPEC section 8.1)
Gradient-boosted classifiers for P(collapse within 12 months) and P(collapse within 24 months), trained on `gold_entity_features`. Labels come from real ASIC register status changes (REGD to EXAD/DRGD). Each model is tracked in MLflow, registered in Unity Catalog as `<catalog>.<schema>.collapse_model_<h>m`, and given the alias `champion` when it beats the minimum AUC.

In [0]:
%pip install -q shap scikit-learn mlflow
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("random_seed", "42")
dbutils.widgets.text("min_auc", "0.65")
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
SEED, MIN_AUC = int(dbutils.widgets.get("random_seed")), float(dbutils.widgets.get("min_auc"))
assert CATALOG and SCHEMA
FQ = f"{CATALOG}.{SCHEMA}"

FEATURES = ["company_age_years", "is_licensee", "ato_debt_latest_aud", "ato_debt_trend_6m_pct", "lodgement_on_time_pct",
            "stp_headcount_trend_pct", "gst_cancelled", "smr_count_24m", "large_flow_smr_count_24m", "apra_flag_count",
            "afca_complaints_latest", "afca_complaints_trend_pct", "breach_reports_12m", "enforcement_alternatives_count",
            "banned_director_count", "director_prior_failed_companies", "related_collapsed_entities"]

In [0]:
import mlflow, numpy as np, pandas as pd, shap
from mlflow.models import infer_signature
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

mlflow.set_registry_uri("databricks-uc")
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/{SCHEMA}_collapse_model")
feat = spark.table(f"{FQ}.gold_entity_features").toPandas()

results = {}
for h in (12, 24):
    df = feat[feat[f"label_collapse_{h}m"].notna()]
    X = df[FEATURES].astype(float).fillna(0.0); y = df[f"label_collapse_{h}m"].astype(int)
    assert y.nunique() == 2, f"{h}m label has a single class"
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
    with mlflow.start_run(run_name=f"collapse_{h}m") as run:
        model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=SEED).fit(Xtr, ytr)
        p = model.predict_proba(Xte)[:, 1]
        auc, ap = roc_auc_score(yte, p), average_precision_score(yte, p)
        mlflow.log_params({"horizon_months": h, "n_train": len(Xtr), "features": ",".join(FEATURES)})
        mlflow.log_metrics({"test_auc": auc, "test_avg_precision": ap, "base_rate": float(y.mean())})
        mlflow.sklearn.log_model(model, name="model", input_example=Xtr.head(5),
                                 signature=infer_signature(Xtr, model.predict_proba(Xtr)[:, 1]),
                                 registered_model_name=f"{FQ}.collapse_model_{h}m")
        imp = pd.Series(np.abs(shap.Explainer(model.predict_proba, Xtr.sample(min(200, len(Xtr)), random_state=SEED))(Xte.head(200)).values[..., 1]).mean(0), index=FEATURES)
        mlflow.log_dict(imp.sort_values(ascending=False).to_dict(), "shap_mean_abs.json")
    results[h] = (auc, ap)
    print(f"{h}m  AUC={auc:.3f}  AP={ap:.3f}  base={y.mean():.3f}")

In [0]:
client = mlflow.MlflowClient()
for h, (auc, _) in results.items():
    name = f"{FQ}.collapse_model_{h}m"
    latest = max(client.search_model_versions(f"name = '{name}'"), key=lambda v: int(v.version))
    if auc >= MIN_AUC:
        client.set_registered_model_alias(name, "champion", latest.version)
        print(f"{name} v{latest.version} -> champion")
    else:
        print(f"{name} v{latest.version} AUC {auc:.3f} < {MIN_AUC}; champion unchanged")
for h in (12, 24):
    client.get_model_version_by_alias(f"{FQ}.collapse_model_{h}m", "champion")  # fails if no champion exists